# Environment Check

Run this notebook to verify that the lab environment is configured correctly before starting the exercises.

## 1. Python Version

In [ ]:
import sys

version = sys.version_info
print(f"Python {version.major}.{version.minor}.{version.micro}")

assert version >= (3, 12), (
    f"Python 3.12+ required, got {version.major}.{version.minor}"
)
print("OK")

## 2. Environment Variables

In [ ]:
import os
from pathlib import Path

# Load .env if present
env_path = Path("..") / ".env"
if env_path.exists():
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip()
    print("Loaded .env file")

required = ["MAAS_API_KEY", "MAAS_BASE_URL"]
optional = {"MAAS_MODEL_ID": "granite-3-2-8b-instruct"}

all_ok = True
for var in required:
    val = os.environ.get(var)
    if val:
        print(f"  {var:<20} OK")
    else:
        print(f"  {var:<20} MISSING")
        all_ok = False

for var, default in optional.items():
    val = os.environ.get(var, default)
    print(f"  {var:<20} {val} {'(default)' if not os.environ.get(var) else ''}")

if all_ok:
    print("\nEnvironment variables OK")
else:
    print("\nSet missing variables before running the lab")

## 3. MaaS Connectivity

In [ ]:
from openai import OpenAI

api_key  = os.environ.get("MAAS_API_KEY")
base_url = os.environ.get("MAAS_BASE_URL")
model_id = os.environ.get("MAAS_MODEL_ID", "granite-3-2-8b-instruct")

if not api_key or not base_url:
    print("Skipping — MAAS_API_KEY or MAAS_BASE_URL not set")
else:
    try:
        client = OpenAI(api_key=api_key, base_url=base_url)
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": "Say hello in one word."}],
            temperature=0.0,
            max_tokens=10
        )
        print(f"Model responded: {response.choices[0].message.content}")
        print("MaaS connectivity OK")
    except Exception as e:
        print(f"Connection failed: {e}")

## 4. ChromaDB Collection

In [ ]:
try:
    import chromadb
    print(f"chromadb version: {chromadb.__version__}")
    
    chroma_client = chromadb.PersistentClient(path="../prebuilt/chroma_db")
    collection = chroma_client.get_collection("basic_fantasy_corpus")
    print(f"Collection: {collection.name}")
    print(f"Documents:  {collection.count()}")
    print("ChromaDB OK")
except ImportError:
    print("chromadb not installed — run: pip install chromadb")
except Exception as e:
    print(f"ChromaDB check failed: {e}")
    print("This is expected if chroma_db has not been built yet.")

## 5. Summary

In [ ]:
import json
from pathlib import Path

prebuilt_files = [
    "eval_results.json",
    "tool_definitions.json",
    "agent_loop_results.json",
]

print("Pre-built files:")
for fname in prebuilt_files:
    fpath = Path("..") / "prebuilt" / fname
    if fpath.exists():
        print(f"  {fname:<30} OK")
    else:
        print(f"  {fname:<30} not yet generated")

print()
print("Environment OK" if all_ok else "Fix the issues above before starting the lab.")